# Punto 7 – Programmatic PySpark Analytical Pipeline (US 4.2)
Reads `s3://stock-big-data-bucket/trusted/stocks_enriched/` Parquet, computes each analytical query in its own independent cell with the PySpark DataFrame API (`groupBy`, `Window`, `avg`, `max`, `rank`), and uploads Parquet outputs to `s3://stock-big-data-bucket/refined/analytics_results/`.

In [ ]:
!pip install -q pyspark boto3

In [ ]:
import getpass
from pathlib import Path
import boto3

BUCKET = "stock-big-data-bucket"
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    aws_access_key_id=getpass.getpass("AWS_ACCESS_KEY_ID: ").strip(),
    aws_secret_access_key=getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip(),
    aws_session_token=getpass.getpass("AWS_SESSION_TOKEN: ").strip(),
)

# shortcut: syncs S3 parquet to local Colab disk via boto3 instead of s3a:// jars, switch to native s3a:// when running on EMR
LOCAL_TRUSTED = Path("/content/data/trusted/stocks_enriched")
PREFIX = "trusted/stocks_enriched/"
downloaded = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/"):
            continue
        dest = LOCAL_TRUSTED / key[len(PREFIX):]
        dest.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, key, str(dest))
        downloaded += 1

assert downloaded > 0, f"No files found under s3://{BUCKET}/{PREFIX}"
print(f"Downloaded {downloaded} files to {LOCAL_TRUSTED}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master("local[*]").appName("Punto7PySparkColab").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
OUTPUT_BASE = Path("/content/data/refined/analytics_results")

base_df = (
    spark.read.parquet(str(LOCAL_TRUSTED))
    .withColumn("ticker", F.trim(F.upper(F.col("ticker"))))
    .withColumn("sector", F.trim(F.col("sector")))
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("close", F.col("close").cast("double"))
    .withColumn("volume", F.col("volume").cast("long"))
    .withColumn("market_cap", F.col("market_cap").cast("double"))
    .filter(
        F.col("ticker").isNotNull()
        & F.col("sector").isNotNull()
        & (F.col("sector") != "__HIVE_DEFAULT_PARTITION__")
        & F.col("date").isNotNull()
        & F.col("close").isNotNull()
    )
    .dropDuplicates(["ticker", "date"])
    .cache()
)
assert base_df.count() > 0, "Empty trusted dataset"


def with_daily_returns(df):
    w = Window.partitionBy("ticker").orderBy("date")
    return df.withColumn("prev_close", F.lag("close", 1).over(w)).withColumn(
        "daily_return_pct",
        F.when(
            (F.col("prev_close").isNotNull()) & (F.col("prev_close") > 0),
            F.round(((F.col("close") - F.col("prev_close")) / F.col("prev_close")) * 100.0, 4),
        ),
    )

### Query 1 — Daily Returns % & `SMA-50` Window Series
**Questions (Q4 & Q5):** What are each stock's daily return %, 50-day Simple Moving Average (`SMA-50`), and % distance from `SMA-50`?

In [ ]:
w_sma50 = Window.partitionBy("ticker").orderBy("date").rowsBetween(-49, 0)

daily_series_df = (
    with_daily_returns(base_df)
    .withColumn("sma_50", F.round(F.avg("close").over(w_sma50), 4))
    .withColumn(
        "pct_vs_sma50",
        F.when(
            (F.col("sma_50").isNotNull()) & (F.col("sma_50") > 0),
            F.round(((F.col("close") - F.col("sma_50")) / F.col("sma_50")) * 100.0, 2),
        ),
    )
    .select("ticker", "sector", "date", "close", "volume", "prev_close", "daily_return_pct", "sma_50", "pct_vs_sma50")
    .orderBy("ticker", "date")
)
daily_series_df.orderBy(F.desc("date"), "ticker").show(15, truncate=False)

# shortcut: coalesce(1) writes a single parquet file per output dataset (~3 MB max), remove coalesce(1) if dataset grows past 1 GB
daily_series_df.coalesce(1).write.mode("overwrite").parquet(str(OUTPUT_BASE / "daily_returns_sma50"))

### Query 2 — Sector Summary (Volume, Market-Cap Rank, Return & Volatility)
**Questions (Q2, Q3 & Q4):** How do sectors compare by trading volume, deduplicated market-cap rank, average daily return %, and daily volatility?

In [ ]:
sector_vol = (
    base_df.filter(F.col("volume").isNotNull())
    .groupBy("sector")
    .agg(
        F.countDistinct("ticker").alias("num_tickers"),
        F.count("*").alias("trading_rows"),
        F.round(F.avg("volume"), 2).alias("avg_daily_volume"),
        F.sum("volume").alias("total_volume"),
    )
)
# Deduplicate market_cap per ticker before summing per sector
sector_mcap = (
    base_df.filter(F.col("market_cap").isNotNull())
    .groupBy("ticker", "sector")
    .agg(F.max("market_cap").alias("market_cap"))
    .groupBy("sector")
    .agg(
        F.count("*").alias("num_companies"),
        F.sum("market_cap").alias("sector_market_cap"),
        F.round(F.avg("market_cap"), 2).alias("avg_company_market_cap"),
    )
)
sector_ret = (
    with_daily_returns(base_df).filter(F.col("daily_return_pct").isNotNull())
    .groupBy("sector")
    .agg(
        F.count("*").alias("return_observations"),
        F.round(F.avg("daily_return_pct"), 4).alias("avg_daily_return_pct"),
        F.round(F.stddev("daily_return_pct"), 4).alias("daily_volatility_pct"),
    )
)
sector_summary_df = (
    sector_vol.join(sector_mcap, "sector", "left")
    .join(sector_ret, "sector", "left")
    .withColumn("market_cap_rank", F.rank().over(Window.orderBy(F.desc("sector_market_cap"))))
    .orderBy("market_cap_rank")
)
sector_summary_df.show(20, truncate=False)
sector_summary_df.coalesce(1).write.mode("overwrite").parquet(str(OUTPUT_BASE / "sector_summary"))

### Query 3 — Top Gainers (Overall & Sector Rank)
**Question (Q1):** Which S&P 500 tickers delivered the highest total return % over the period (overall and within each sector)?

In [ ]:
w_asc = Window.partitionBy("ticker").orderBy(F.col("date").asc())
w_desc = Window.partitionBy("ticker").orderBy(F.col("date").desc())

top_gainers_df = (
    base_df
    .withColumn("rn_first", F.row_number().over(w_asc))
    .withColumn("rn_last", F.row_number().over(w_desc))
    .filter((F.col("rn_first") == 1) | (F.col("rn_last") == 1))
    .groupBy("ticker")
    .agg(
        F.max("sector").alias("sector"),
        F.min(F.when(F.col("rn_first") == 1, F.col("date"))).alias("start_date"),
        F.max(F.when(F.col("rn_last") == 1, F.col("date"))).alias("end_date"),
        F.round(F.max(F.when(F.col("rn_first") == 1, F.col("close"))), 4).alias("first_close"),
        F.round(F.max(F.when(F.col("rn_last") == 1, F.col("close"))), 4).alias("last_close"),
    )
    .filter(F.col("first_close") > 0)
    .withColumn(
        "total_return_pct",
        F.round(((F.col("last_close") - F.col("first_close")) / F.col("first_close")) * 100.0, 2),
    )
    .withColumn("overall_rank", F.rank().over(Window.orderBy(F.desc("total_return_pct"))))
    .withColumn("sector_rank", F.rank().over(Window.partitionBy("sector").orderBy(F.desc("total_return_pct"))))
    .orderBy("overall_rank")
)
top_gainers_df.show(15, truncate=False)
top_gainers_df.coalesce(1).write.mode("overwrite").parquet(str(OUTPUT_BASE / "top_gainers"))

### Query 4 — Volume Leaders by Sector
**Question (Q2):** Which stocks are the top trading-volume leaders within each sector?

In [ ]:
volume_leaders_df = (
    base_df.filter(F.col("volume").isNotNull())
    .groupBy("sector", "ticker")
    .agg(
        F.count("*").alias("trading_days"),
        F.round(F.avg("volume"), 2).alias("avg_daily_volume"),
        F.max("volume").alias("max_daily_volume"),
        F.sum("volume").alias("total_volume"),
    )
    .withColumn(
        "sector_volume_rank",
        F.rank().over(Window.partitionBy("sector").orderBy(F.desc("avg_daily_volume"))),
    )
    .orderBy("sector", "sector_volume_rank")
)
volume_leaders_df.filter(F.col("sector_volume_rank") <= 3).show(40, truncate=False)
volume_leaders_df.coalesce(1).write.mode("overwrite").parquet(str(OUTPUT_BASE / "volume_leaders_by_sector"))

### Upload Refined Parquet Outputs to S3

In [ ]:
S3_PREFIX = "refined/analytics_results"
uploaded = 0
for path in OUTPUT_BASE.rglob("*"):
    if path.is_file() and not path.name.endswith(".crc"):
        key = f"{S3_PREFIX}/{path.relative_to(OUTPUT_BASE).as_posix()}"
        s3.upload_file(str(path), BUCKET, key)
        uploaded += 1

assert uploaded >= 2, f"Expected at least 2 files (_SUCCESS + parquet), uploaded {uploaded}"
print(f"Uploaded {uploaded} files to s3://{BUCKET}/{S3_PREFIX}/")